
# Stellar mass-to-light ratios vs SSP age, per band

``M_★/L_band`` rises with stellar age in every band; the rise is
steepest in g, where massive young stars dominate, and shallowest in
``K_s``, where red giants contribute at every age past the first
~100 Myr.

A narrow burst centred at a sweep of lookback times approximates an
SSP. ``M_★`` comes from integrating the recovered SFH; ``L_band``
from inverting tengri's photometric prediction back to ``L_ν``
(using a fixed ``d_L`` at ``z = 0.01``) and multiplying by the
band's effective frequency.


In [ ]:
import warnings

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np

import tengri
from tengri.analysis.plotting import setup_style

setup_style()
warnings.filterwarnings("ignore", message=".*BakedInBackend.*")

L_SUN_ERG_S = 3.839e33
D_L_CM = 43.6 * 3.086e24   # luminosity distance at z = 0.01

BANDS = [
    ("sdss_g",   "g (SDSS)"),
    ("sdss_r",   "r (SDSS)"),
    ("sdss_i",   "i (SDSS)"),
    ("2mass_ks", "K_s (2MASS)"),
]
COLORS = plt.cm.viridis(np.linspace(0.05, 0.92, len(BANDS)))

obs = tengri.Observation(
    photometry=tengri.Photometry.from_names([b for b, _ in BANDS])
)
nu_eff = 2.998e18 / np.array(
    [float(jnp.mean(w)) for w in obs.photometry.filter_waves]
)

model = tengri.SEDModel.build(
    tengri.load_ssp(),
    observation=obs,
    sfh={"type": "tsnorm", "*": tengri.FIXED,
         "peak_lbt_gyr": tengri.Uniform(0.03, 13.0),
         "width_gyr": 0.05, "log_peak_sfr": 1.0,
         "skew": 0.0, "trunc": 13.0},
    dust={"type": "two_component", "*": tengri.FIXED,
          "tau_diff": 0.0, "tau_bc": 0.0},
    redshift=tengri.Fixed(0.01),
)
baseline = dict(model.spec.sample(jax.random.PRNGKey(0)))

# Past ~1 Gyr the narrow burst window starts to clip against the
# universe age and the SFH normalisation goes noisy.
ages = np.geomspace(0.03, 1.0, 16)
ml = np.empty((len(BANDS), ages.size))

for j, age in enumerate(ages):
    p = {**baseline, "sfh_tsnorm_peak_lbt_gyr": jnp.float64(age)}
    sfh = model.predict_sfh(p)
    m_star = float(np.trapezoid(np.asarray(sfh["sfr_mean"]),
                                np.asarray(sfh["t_gyr"]) * 1e9))
    flux = np.asarray(model.predict_photometry(p))
    L_band = flux * 4 * np.pi * D_L_CM**2 * nu_eff / L_SUN_ERG_S
    ml[:, j] = m_star / np.maximum(L_band, 1e-12)

fig, ax = plt.subplots(figsize=(7.0, 4.6))
for (_, label), color, m_l in zip(BANDS, COLORS, ml):
    ax.loglog(ages, m_l, color=color, lw=1.6, label=label)
ax.set(xlabel="Stellar burst age  [Gyr]",
       ylabel=r"$M_\star\,/\,L$  [$M_\odot\,/\,L_\odot$]",
       ylim=(0.03, 3.0))
ax.legend(frameon=False, fontsize=9, loc="upper left")

fig.tight_layout()
fig.savefig("plot_mass_to_light_ratios.png", dpi=150, bbox_inches="tight")